# 第 12 章　変数間の関係

明日の夕方は風が強そうです。「風の強い時間ほど自転車の貸出が多い」という全時刻の結果を，17 時台の準備にも使えるでしょうか。このノートでは，風速と貸出件数の関係が，比べる時間帯によってどう変わるかを調べます。

講義では全時刻と 8 時台・13 時台を比べ，相関の違いを時刻別の平均から考えます。実習では 17 時台を自分で取り出し，夕方の準備をする担当者へ，どの記録から何が分かったかと，さらに知りたい条件を伝えます。

前半は，章本文の説明に合わせて，上から順にセルを実行します。コードの前に何を調べるかを読み，実行後は表や図のどの値が問いへの答えになるかを確かめてください。後半の約 20 分の実習では，比べる対象や条件を自分で選び，結果を短い文章にします。前半のコードは実行できる見本なので，作図の処理をすべて書き写す必要はありません。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV は，手元にファイルがなければ公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

file_name = "bike-sharing-hourly.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

hourly = pd.read_csv(csv_path)
hourly[["dteday", "hr", "windspeed", "cnt"]].head(3)

## 本文と一緒に実行

### 1. 同じ風速でも貸出件数は同じか

まず全時刻の記録を使い，風速指標を横軸，貸出件数を縦軸に置きます。一点は，ある日付のある時間帯の記録です。

風速指標が 0.1940 の記録から，貸出件数が最も少ない行と多い行を選び，図で強調します。同じ横位置にある二点が，どの日付・時間帯を表すかを表で確かめてください。最後に全時刻の相関を求め，図の点の広がりと合わせて読みます。

In [ ]:
columns = ["dteday", "hr", "windspeed", "cnt"]
same_wind = hourly.loc[hourly["windspeed"] == 0.1940]
low_rentals = same_wind.loc[same_wind["cnt"].idxmin()]
high_rentals = same_wind.loc[same_wind["cnt"].idxmax()]

In [ ]:
plt.rcParams["figure.figsize"] = (6.1, 3.7)
fig, ax = plt.subplots(layout="constrained")
ax.scatter(hourly["windspeed"], hourly["cnt"], s=8, color="#286b70",
           alpha=0.15, linewidths=0)
ax.scatter(low_rentals["windspeed"], low_rentals["cnt"], s=58, color="#a34f30",
           marker="o", edgecolors="white", linewidths=1.2, zorder=3, clip_on=False)
ax.scatter(high_rentals["windspeed"], high_rentals["cnt"], s=58, color="#a34f30",
           marker="s", edgecolors="white", linewidths=1.2, zorder=3)
ax.set(xlabel="Wind-speed index", ylabel="Rentals per hour",
       xlim=(-0.025, 0.875), ylim=(0, 1000))
plt.show()

In [ ]:
print(len(same_wind))
same_wind.loc[[low_rentals.name, high_rentals.name], columns]

In [ ]:
overall_r = hourly["windspeed"].corr(hourly["cnt"])
print(f"{overall_r:+.3f}")

風速指標が同じ 0.1940 の記録は 1,657 行あり，貸出件数は 1〜943 件に広がっています。風速が同じでも，その時間帯に何件借りられたかは大きく違います。

全時刻の相関は +0.093 です。弱い正の直線的な関連があるという結果と，一つの日の貸出件数を風速から正確に予想できるかは別の問題です。図の縦の広がりを使って，その違いを説明してください。

### 2. 四つの点で相関の意味を考える

全時刻の相関が正だったのは，二つの列の値がどのように組み合わさったためでしょうか。本文と同じ教材用の 4 点で，相関の計算を確かめます。

左の図は `x` と `y`，右の図は `x` と `repaired_y` の組です。それぞれの列に含まれる値は同じですが，どの `x` とどの `y` を一行に組み合わせるかが違います。二つの図の相関が同じになるかを予想してください。

In [ ]:
small = pd.DataFrame({"x": [1, 2, 4, 5], "y": [1, 4, 2, 5], "repaired_y": [5, 4, 2, 1]})
small

In [ ]:
plt.rcParams["figure.figsize"] = (6.3, 3.2)
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, layout="constrained")
for ax, column, title in zip(axes, ["y", "repaired_y"], ["Original pairs", "Different pairs"]):
    ax.axhline(3, color="#6f7b86", linestyle="--", linewidth=1)
    ax.axvline(3, color="#6f7b86", linestyle="--", linewidth=1)
    ax.scatter(small["x"], small[column], color="#286b70", s=35, zorder=3)
    for name, x, y in zip("ABCD", small["x"], small[column]):
        ax.annotate(name, (x, y), xytext=(5, 4), textcoords="offset points")
    ax.set(title=title, xlabel="x", xlim=(0.4, 5.7), ylim=(0.4, 5.7),
           xticks=[1, 2, 3, 4, 5], yticks=[1, 2, 3, 4, 5])
axes[0].set_ylabel("y")
plt.show()

### 平均からのずれを標準偏差で割る

相関を求めるため，まず各値が平均からどれくらい離れているかを計算します。`x` の平均からのずれを足すと，−2 − 1 + 1 + 2 = 0 となり，散らばりの大きさを表せません。そこで，ずれを二乗した 4，1，1，4 の平均 2.5 を求め，平方根を取ります。この約 1.58 が，この 4 個の値の標準偏差です。ここでは二乗したずれの合計を 4 で割っています。

続いて，それぞれの列で「平均からのずれ ÷ 標準偏差」を求め，二列の結果を同じ行どうしでかけます。この積の平均が相関係数です。片方の測定の単位を替えて値が 10 倍になっても，ずれと標準偏差は両方とも 10 倍になるので，割った値は変わりません。

次のセルで，標準偏差，4 個の積，その平均を順に確認します。`** 2` は二乗，`** 0.5` は平方根です。その後の `.corr()` の結果と，手順を分けて求めた相関が一致するかを確かめてください。

In [ ]:
dx = small["x"] - small["x"].mean()
dy = small["y"] - small["y"].mean()
sd_x = (dx ** 2).mean() ** 0.5
sd_y = (dy ** 2).mean() ** 0.5
standardized_products = (dx / sd_x) * (dy / sd_y)
print("Standard deviations:", round(sd_x, 3), round(sd_y, 3))
print("Products:", standardized_products.round(2).to_list())
print("Mean product:", round(standardized_products.mean(), 2))


In [ ]:
print(round(small["x"].corr(small["y"]), 2))
print(round(small["x"].corr(small["repaired_y"]), 2))

左の組み合わせの相関は +0.60，右は −1.00 です。どちらの図でも各列の平均は 3 ですが，平均より大きい値と小さい値の組み合わせが変わりました。

平均 3 の縦線・横線を基準に，両方のずれが正または両方とも負の点と，一方だけが正の点を探してください。本文の 4 点の表と照らし合わせると，点の数だけでなく，ずれの積の大きさが相関の符号に関わることを説明できます。

### 3. 同じ時間帯に限定して比べる

全時刻の図には，貸出の少ない早朝と多い夕方など，時間帯の違いも含まれます。まず 8 時台と 13 時台をそれぞれ取り出し，異なる日付の同じ時間帯どうしを比べます。

実行前に，全体で正の関連が見えた理由を考えます。「同じ時間帯でも風速が大きい日は貸出が多い」なら，時間帯を限定した図にも正の関連が見られそうです。「平均の風速も貸出も小さい早朝と，両方が大きい夕方などを混ぜたことが関わる」なら，限定後には正の関連が弱まるか見えなくなるかもしれません。どの結果なら自分の説明を見直すかを先に言葉にします。

行数と相関を求めた後，全時刻の図と同じ軸の範囲で散布図を描きます。例えば 17 時台の記録はどちらの図にも入りません。どの記録が残り，どの記録が外れた比較なのかを確認してから，相関と点の広がりを比べてください。

In [ ]:
at_eight = hourly.loc[hourly["hr"] == 8]
at_thirteen = hourly.loc[hourly["hr"] == 13]

r_eight = at_eight["windspeed"].corr(at_eight["cnt"])
r_thirteen = at_thirteen["windspeed"].corr(at_thirteen["cnt"])
print(f"08:00: {len(at_eight)} rows, r={r_eight:+.3f}")
print(f"13:00: {len(at_thirteen)} rows, r={r_thirteen:+.3f}")

In [ ]:
plt.rcParams["figure.figsize"] = (7, 3.4)
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, layout="constrained")
for ax, selected, title in zip(axes, [at_eight, at_thirteen], ["08:00", "13:00"]):
    ax.scatter(selected["windspeed"], selected["cnt"],
               color="#286b70", alpha=0.36, s=13, linewidths=0)
    ax.set(title=title, xlabel="Wind-speed index", xlim=(-0.025, 0.875),
           ylim=(0, 1000), yticks=[0, 250, 500, 750, 1000])
axes[0].set_ylabel("Rentals per hour")
plt.show()

8 時台は 727 行で相関が約 −0.092，13 時台は 729 行で約 −0.086 です。全時刻の正の相関から符号が変わりましたが，どちらも 0 に近い値で，直線的な関連は弱いままです。

図には，同じ風速指標でも貸出件数が大きく違う点が残っています。「時刻を限定したら相関の符号が変わった」という結果と，「風速だけでその日の件数を決められる」という主張を区別してください。比較前に考えた説明と照らし，この二つの時間帯の結果に合う点と合わない点を述べてください。時刻を混ぜたことが関わるという説明をさらに確かめるため，次は時刻別の平均を調べます。

### 4. 時刻ごとの平均で全体の関連を考える

全時刻と同じ時間帯だけの比較で，相関の符号が違いました。その理由を考えるため，各時間帯の風速指標と貸出件数を，それぞれ複数の日にわたって平均します。

次の図の「05」や「17」は，ある一日の記録ではなく，多くの日の 5 時台や 17 時台をまとめた平均点です。早朝から夕方へ，二つの平均がどちらへ変わるかを見てください。各日の点を一枚にまとめた図に，この時刻による違いも含まれていたことが分かります。

In [ ]:
hour_means = hourly.groupby("hr")[["windspeed", "cnt"]].mean()
hour_means.loc[[5, 8, 13, 16, 17, 23]].round(3)

In [ ]:
plt.rcParams["figure.figsize"] = (6, 3.8)
fig, ax = plt.subplots(layout="constrained")
ax.scatter(hour_means["windspeed"], hour_means["cnt"], s=28, color="#667f8d")
for hour, offset in {
    5: (-18, 5), 8: (-7, 8), 13: (-4, 11), 16: (5, 6), 17: (6, -2), 23: (6, -2),
}.items():
    mean_wind = hour_means.loc[hour, "windspeed"]
    mean_rentals = hour_means.loc[hour, "cnt"]
    ax.scatter(mean_wind, mean_rentals, marker="s", s=32, color="#a34f30", zorder=3)
    ax.annotate(f"{hour:02d}", (mean_wind, mean_rentals), xytext=offset,
                textcoords="offset points")
ax.set(
    xlabel="Mean wind-speed index", ylabel="Mean rentals per hour",
    xlim=(0.137, 0.252), ylim=(-20, 500),
)
plt.show()

In [ ]:
hour_mean_r = hour_means["windspeed"].corr(hour_means["cnt"])
print(f"Hourly averages: {len(hour_means)} points, r={hour_mean_r:+.3f}")

24 個の時刻別平均の相関は約 +0.817 です。この係数が表すのは，風速の平均が高い時間帯ほど貸出件数の平均も多いという，時間帯どうしの関連です。

平均点だけでは，同じ時間帯の日ごとの件数の広がりは見えません。毎日の 8 時台の貸出を予想したいときに，この係数をそのまま予測の強さと読んでよいか，先ほどの 727 行の図と比べて説明してください。

### 5. 同じ時刻でも違う日を調べる

同じ時刻の記録だけにしても，貸出件数の違いは大きく残りました。第 11 章で見たように，土日・祝日かどうかも関係しそうです。8 時台と 13 時台のそれぞれで，この二種類の日を色と形で分けます。

`workingday` は土日・祝日なら `0`，それ以外なら `1` です。群ごとの行数と平均を表で読み，図では各群の点の高さと広がりを確認してください。

In [ ]:
calendar_means = (
    hourly.loc[hourly["hr"].isin([8, 13])]
    .groupby(["hr", "workingday"])["cnt"]
    .agg(rows="size", mean="mean")
)
calendar_means.round(1)

In [ ]:
plt.rcParams["figure.figsize"] = (7, 3.8)
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, layout="constrained")
for ax, selected, title in zip(axes, [at_eight, at_thirteen], ["08:00", "13:00"]):
    for flag, label, marker, color in [
        (0, "Weekend / holiday", "s", "#a34f30"),
        (1, "Other weekday", "o", "#286b70"),
    ]:
        group = selected.loc[selected["workingday"] == flag]
        ax.scatter(group["windspeed"], group["cnt"], label=label,
                   color=color, marker=marker, alpha=0.38, s=13, linewidths=0)
    ax.set(title=title, xlabel="Wind-speed index", xlim=(-0.025, 0.875),
           ylim=(0, 1000), yticks=[0, 250, 500, 750, 1000])
axes[0].set_ylabel("Rentals per hour")
axes[1].legend(frameon=False, loc="upper right", fontsize=8, markerscale=1.4)
plt.show()

8 時台では土日・祝日以外の平均が高く，13 時台では土日・祝日の平均が高くなっています。風速と時間帯が同じでも，どの種類の日かによって貸出件数が違いそうです。

冒頭の夕方の準備を考える場合も，全時刻の相関だけで判断せず，その時間帯に当てはまる日の記録を調べる必要があります。実習では 17 時台を取り出して，ここまでの比較を自分で行います。

## Jupyter 実習（約 20 分）

夕方の 17 時台に備える担当者へ，全時刻で見えた風速と貸出件数の関係をそのまま伝えてよいかを調べます。講義の 8 時台・13 時台から対象を替え，自分で選んだ 17 時台の記録を `chosen`，その相関を `chosen_r` に入れます。

全時刻との違いだけでなく，同じ風速でも件数がどれくらい違うかを図で確かめ，夕方の準備にさらに必要な情報を考えてください。

## 1. 全時刻の図から予想する（約 5 分）

動く例として，収録された全時刻を一枚の散布図にします。横軸は風速指標，縦軸は貸出件数です。全体の相関も表示します。

夕方の準備を考えるために 17 時台を選ぶ理由を一文で書きます。17 時台だけにすると，点が集まる高さや，同じ風速での件数の幅，相関の符号はどう変わりそうでしょうか。結果を見る前に，その予想のもとになる説明と，図で確かめたいところを一つ決めます。例えば，正の関連に時間帯の混合が関わると考えるなら，17 時台に限定したときに相関がどうなりそうかを予想できます。例えば，風速指標 0.1940 付近の点が縦にどれくらい広がるかを比べられます。

（ここに理由と予想を記入）

In [ ]:
overall_r = hourly["windspeed"].corr(hourly["cnt"])
print(f"All hours: {len(hourly):,} rows, r={overall_r:+.3f}")

x_limits = (-0.025, 0.875)
y_limits = (0, 1000)
fig, ax = plt.subplots(layout="constrained")
ax.scatter(hourly["windspeed"], hourly["cnt"], color="#286b70",
           alpha=0.15, s=8, linewidths=0)
ax.set(xlabel="Wind-speed index", ylabel="Rentals per hour",
       xlim=x_limits, ylim=y_limits)
plt.show()

## 2. 17 時台の記録で比べる（約 9 分）

最初の課題セルで，`hourly` の `hr` が `target_hour` に等しい行を選び，`chosen` に入れます。`target_hour` は 17 です。抽出後の行数と先頭の 3 行を見て，日付は異なっても，各行が 17 時台を表すことを確かめてください。

次の課題セルでは，`chosen` にある風速指標と貸出件数の相関を求め，`chosen_r` に入れます。同じ行の二つの値を対応させて計算する点は，講義の 4 点の例と同じです。

抽出と相関の二行を記入すると，全時刻と 17 時台の図が同じ軸で表示されます。最初に決めた位置を比べ，点の高さや広がりが予想どおり変わったかを確かめます。

In [ ]:
target_hour = 17
chosen = None
# target_hour に合う行を選び，chosen に入れる。

if chosen is None:
    print("時刻を指定する一行を書いてから，このセルを実行します。")
else:
    print(f"{len(chosen):,} rows at {target_hour:02d}:00")
    display(chosen[["dteday", "hr", "windspeed", "cnt"]].head(3))

In [ ]:
chosen_r = None
if chosen is not None:
    # chosen の同じ行の二列の相関を chosen_r に入れる。
    pass

if chosen_r is None:
    print("時刻の選択と相関の両方を書くと，比較図が出ます。")
else:
    print(f"All hours: r={overall_r:+.3f}")
    print(f"{target_hour:02d}:00: r={chosen_r:+.3f}")
    fig, axes = plt.subplots(
        1, 2, figsize=(8, 3.4), sharex=True, sharey=True, layout="constrained"
    )
    for ax, data, title, alpha, size in [
        (axes[0], hourly, "All hours", 0.15, 8),
        (axes[1], chosen, f"{target_hour:02d}:00", 0.36, 13),
    ]:
        ax.scatter(data["windspeed"], data["cnt"], color="#286b70",
                   alpha=alpha, s=size, linewidths=0)
        ax.set(title=title, xlabel="Wind-speed index", xlim=x_limits, ylim=y_limits)
    axes[0].set_ylabel("Rentals per hour")
    plt.show()

<details>
<summary>二つの行の書き方を確認する</summary>

まず講義の「同じ時間帯に限定して比べる」で，時間帯を選んだ行を読み返してください。書き方を確認したい場合は，次の例を使います。`chosen = None` の下に，対象時刻に当てはまる行を選ぶ行を書きます。

```python
chosen = hourly.loc[hourly["hr"] == target_hour]
```

`if chosen is not None:` の中で，次の一行を書きます。

```python
    chosen_r = chosen["windspeed"].corr(chosen["cnt"])
```

</details>

## 3. 夕方の貸出について報告する（約 6 分）

最初に考えた説明と予想を，図と相関の結果に照らします。その説明を保てる点と，改める点を含め，夕方の準備をする担当者に 3〜4 文で伝えます。

1. 全時刻と 17 時台では，比較に使った行数と相関がどう違ったか。それぞれにどの時間帯の記録が含まれるか。
2. 17 時台の点の集まりで，同じくらいの風速でも件数がどの程度違うか。図で読んだ位置か範囲を一つ挙げる。
3. この結果だけでは明日の件数まで決められない理由と，曜日や年などをどう分ければ，次にどの説明を確かめられるかを一つ述べる。

（ここに報告を記入）

カーネルを再起動し，上から全セルを実行します。図と説明を残したノートブックを提出します。

### 土日・祝日以外の 17 時台（任意）

同じ `chosen` から `workingday` が `1` の行に絞ります。17 時台の記録を土日・祝日以外の日に限ると，どのような日付が比較に残りますか。行数と相関を調べ，必須課題で作った 17 時台の説明にどんな情報を添えられるか考えます。

### 出典

Capital Bikeshare の時間別集計と気象と曜日・祝日の情報は，Hadi Fanaee-T が UCI Machine Learning Repository で公開した `hour.csv` を教材用に改名したものです。 [データ配布元](https://doi.org/10.24432/C5W894) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)